In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('..')

In [3]:
from Utils.Accuracy_measures import topk_accuracy
from Utils.TinyImageNet_loader import get_tinyimagenet_dataloaders
from Utils.num_parameter import count_parameters
from Models.Resnet50 import Resnet50

import torchvision.transforms as transforms
from torch import nn
from torch import optim

import time
import torch
import os

In [18]:
device = 'cpu'

In [4]:
# Set up the transforms and train/test loaders
image_size = 192

tiny_transform_train = transforms.Compose([
            transforms.RandomHorizontalFlip(),
            transforms.RandomCrop(64, padding=4),
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
tiny_transform_val = transforms.Compose([
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])
tiny_transform_test = transforms.Compose([
            transforms.Resize((image_size, image_size)), 
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
        ])


train_loader, test_loader, _ = get_tinyimagenet_dataloaders(
                                                        data_dir = '../datasets',
                                                        transform_train=tiny_transform_train,
                                                        transform_val=tiny_transform_val,
                                                        transform_test=tiny_transform_test,
                                                        batch_size=64,
                                                        image_size=192)

In [6]:
model= Resnet50(pretrained=True,
                          weights_path='../weights/resnet50_weights.pth',
                          input_shape=(192,192),
                          num_classes=10,
                          avg_pool=False,
                          new_classifier=None)
    
num_parameters = count_parameters(model)
classifier_parameters = count_parameters(model.fc)
print(f'This Model has {num_parameters} parameters')
print(f'This Model has {classifier_parameters} classifier parameters')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

This Model has 24245322 parameters
This Model has 737290 classifier parameters


In [15]:
layer = 0
for child in model.children():
    layer+=1
    if layer < 10:
        for param in child.parameters():
            param.requires_grad = False

In [16]:
num_parameters = count_parameters(model)
classifier_parameters = count_parameters(model.fc)
print(f'This Model has {num_parameters} parameters')
print(f'This Model has {classifier_parameters} classifier parameters')

This Model has 24245322 parameters
This Model has 737290 classifier parameters


In [21]:
# Define train and test functions (use examples)
def train_epoch(loader, epoch):
        model.train()
    
        start_time = time.time()
        running_loss = 0.0
        correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

        for _, (inputs, targets) in enumerate(loader):
            print(_)
            inputs, targets = inputs.to(device), targets.to(device)
        
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
            for k in accuracies:
                correct[k] += accuracies[k]['correct']

        elapsed_time = time.time() - start_time
        top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
        avg_loss = running_loss / len(loader.dataset)
    
        report_train = f'Train epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
        print(report_train)

        return report_train

def test_epoch(loader, epoch):
        model.eval()
    
        start_time = time.time()
        running_loss = 0.0
        correct = {1:0.0, 2:0.0, 3:0.0, 4:0.0, 5:0.0} # set the initial correct count for top1-to-top5 accuracy

        for _, (inputs, targets) in enumerate(loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)

            running_loss += loss.item()
            accuracies = topk_accuracy(outputs, targets, topk=(1, 2, 3, 4, 5))
            for k in accuracies:
                correct[k] += accuracies[k]['correct']

        elapsed_time = time.time() - start_time
        top1_acc, top2_acc, top3_acc, top4_acc, top5_acc = [(correct[k]/len(loader.dataset)) for k in correct]
        avg_loss = running_loss / len(loader.dataset)
    
        report_test = f'Test epoch {epoch}: top1={top1_acc}%, top2={top2_acc}%, top3={top3_acc}%, top4={top4_acc}%, top5={top5_acc}%, loss={avg_loss}, time={elapsed_time}s'
        print(report_test)

        return report_test

In [13]:
TEST_ID = 'Test_ID003'
result_dir = os.path.join('../results', TEST_ID)
result_subdir = os.path.join(result_dir, 'accuracy_stats')
model_subdir = os.path.join(result_dir, 'model_stats')

os.makedirs(result_subdir, exist_ok=True)
os.makedirs(model_subdir, exist_ok=True)
    
with open(os.path.join(result_dir, 'model_stats', 'model_info.txt'), 'a') as f:
        f.write(f'total number of parameters:\n{num_parameters}\ntotal number of classifier parameters{classifier_parameters}')

In [23]:
for _,(x,y) in enumerate(train_loader):
    break

y = y.to(device)
x = x.to(device)

out = model(x)

In [24]:
out.shape

torch.Size([64, 10])

In [22]:
n_epoch = 2
print(f'Starts training for {len(range(n_epoch))} epochs\n')
for epoch in range(1,n_epoch+1):
        report_train = train_epoch(train_loader, epoch)
        # report_test = test_epoch(test_loader, epoch)
    
        # report = report_train + '\n' + report_test + '\n\n'
        # if epoch % 10 == 0:
        #     model_path = os.path.join(result_dir, 'model_stats', f'Model_epoch_{epoch}.pth')
        #     torch.save(model.state_dict(), model_path)
        # with open(os.path.join(result_dir, 'accuracy_stats', 'report.txt'), 'a') as f:
        #     f.write(report)
    

Starts training for 2 epochs

0


IndexError: Target 142 is out of bounds.